In [1]:

import subprocess, sys, time, os

_missing = []

def _need(pkg_import, pkg_install):
    try:
        __import__(pkg_import)
    except ImportError:
        _missing.append(pkg_install)

_need('onnxruntime', 'onnxruntime')
_need('pyarrow',     'pyarrow>=12.0')

if any('onnxruntime' in p for p in _missing):
    _missing.append('protobuf>=3.20.3,<7.0.0')

if _missing:
    print(f'Instalando: {_missing}')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + _missing,
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print('pip stderr:', r.stderr[-800:])
        raise RuntimeError('Instalación fallida — revisar pip stderr arriba')
    print(f'✅ Instalados: {_missing}')
else:
    print('✅ Todas las dependencias ya disponibles (onnxruntime, pyarrow)')

_health = subprocess.run(
    ['bash', '-c', 'curl -sf http://localhost:6000/health'],
    capture_output=True, text=True
)
if _health.returncode != 0:
    print('Iniciando MLflow...', end=' ', flush=True)
    subprocess.Popen(
        ['bash', '/srv/start_mlflow.sh'],
        stdout=open('/tmp/mlflow.log', 'w'),
        stderr=subprocess.STDOUT
    )
    time.sleep(6)
    print('✅ http://localhost:6000')
else:
    print('✅ MLflow ya activo — http://localhost:6000')

print('\nEntorno OE3-A listo — continuar con Cell 1')

✅ Todas las dependencias ya disponibles (onnxruntime, pyarrow)
Iniciando MLflow... ✅ http://localhost:6000

Entorno OE3-A listo — continuar con Cell 1


In [2]:
import sys, os, json, hashlib, pickle, time, importlib.util
sys.path.insert(0, '/srv/python_packages')
from pathlib import Path
import numpy as np
import pandas as pd

import sklearn
import lightgbm as lgb
import onnxruntime as rt
import mlflow

mlflow.set_tracking_uri('http://localhost:6000')
mlflow.set_experiment('latam2cloud')

MODELS_DIR   = Path('/srv/work/models')
WORK_DIR     = Path('/srv/work')
WRAPPER_PATH = MODELS_DIR / 'ensemble_d38_infer.py'
APPLY_PATH   = MODELS_DIR / 'dsdl_apply_d38.py'
CLASS_NAMES  = ['Normal', 'Baja', 'Media', 'Alta']
W_RF, W_LGB  = 0.80, 0.20

MODEL_NAME   = 'latam2cloud_monitor'
SCRIPT_PATH  = str(APPLY_PATH)
SPLUNK_HOST  = 'latam2cloud.iartificial.io'
SPLUNK_PORT  = 8089

print(f'sklearn       : {sklearn.__version__}')
print(f'onnxruntime   : {rt.__version__}')
print(f'Models dir    : {MODELS_DIR}  exists: {MODELS_DIR.exists()}')
print(f'Model en Splunk: {MODEL_NAME}')
print(f'Script DSDL   : {SCRIPT_PATH}')

sklearn       : 1.9.1
onnxruntime   : 1.31.0
Models dir    : /srv/work/models  exists: True
Model en Splunk: latam2cloud_monitor
Script DSDL   : /srv/work/models/dsdl_apply_d38.py


In [3]:

def sha256_short(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        h.update(f.read())
    return h.hexdigest()[:16]

artifacts = {
    'ensemble_d38_infer.py'      : MODELS_DIR / 'ensemble_d38_infer.py',
    'lgb_v7.onnx'                : MODELS_DIR / 'lgb_v7.onnx',
    'rf_v51_model.pkl'           : MODELS_DIR / 'rf_v51_model.pkl',
    'scaler_v5.pkl'              : MODELS_DIR / 'scaler_v5.pkl',
    'feature_cols_v5.json'       : MODELS_DIR / 'feature_cols_v5.json',
    'ensemble_d38_manifest.json' : MODELS_DIR / 'ensemble_d38_manifest.json',
    'dsdl_apply_d38.py'          : MODELS_DIR / 'dsdl_apply_d38.py',
}

print('=== Verificación artefactos D40 ===')
all_present = True
for name, path in artifacts.items():
    exists = path.exists()
    size   = f'{path.stat().st_size / 1024:.1f} KB' if exists else 'FALTA'
    status = '✅' if exists else '❌'
    print(f'  {status} {name:<35} {size}')
    all_present = all_present and exists

print()
if all_present:
    with open(MODELS_DIR / 'ensemble_d38_manifest.json') as f:
        manifest = json.load(f)
    sha_rf  = sha256_short(MODELS_DIR / 'rf_v51_model.pkl')
    sha_lgb = sha256_short(MODELS_DIR / 'lgb_v7.onnx')
    ok_rf  = sha_rf  == manifest['models']['rf']['sha256']
    ok_lgb = sha_lgb == manifest['models']['lgb']['sha256']
    print(f'Manifest ensemble : {manifest["ensemble"]}  decisión: {manifest["decision"]}')
    print(f'Checksum RF       : {sha_rf}  {"✅" if ok_rf  else "❌ MISMATCH"}')
    print(f'Checksum LGB ONNX : {sha_lgb}  {"✅" if ok_lgb else "❌ MISMATCH"}')
    if not (ok_rf and ok_lgb):
        raise RuntimeError('Checksum mismatch — modelos no corresponden al manifest D40')
    print('\n✅ Todos los artefactos presentes y verificados')
else:
    print('❌ Artefactos faltantes — ejecutar nb08 y nb09 primero')

=== Verificación artefactos D40 ===
  ✅ ensemble_d38_infer.py               3.5 KB
  ✅ lgb_v7.onnx                         37110.2 KB
  ✅ rf_v51_model.pkl                    3855877.4 KB
  ✅ scaler_v5.pkl                       2.8 KB
  ✅ feature_cols_v5.json                1.7 KB
  ✅ ensemble_d38_manifest.json          3.6 KB
  ✅ dsdl_apply_d38.py                   2.7 KB

Manifest ensemble : D38  decisión: D40
Checksum RF       : 1cc0a34206f4c5b8  ✅
Checksum LGB ONNX : 9524180dad737af6  ✅

✅ Todos los artefactos presentes y verificados


In [4]:

if not APPLY_PATH.exists():
    print('dsdl_apply_d38.py no encontrado — regenerando desde nb09 lógica...')
    APPLY_CODE = '''
"""dsdl_apply_d38.py — Splunk AI Toolkit (DSDL) integration · Ensemble D38 D40"""
import pickle, importlib.util, json
import numpy as np
import pandas as pd
from pathlib import Path

MODELS_DIR   = Path("/srv/work/models")
WRAPPER_PATH = MODELS_DIR / "ensemble_d38_infer.py"
_infer = _scaler = _fcols = None

def _load():
    global _infer, _scaler, _fcols
    if _infer is None:
        spec = importlib.util.spec_from_file_location("ensemble_d38_infer", WRAPPER_PATH)
        mod  = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
        _infer = mod
    if _scaler is None:
        with open(MODELS_DIR / "scaler_v5.pkl", "rb") as f: _scaler = pickle.load(f)
    if _fcols is None:
        with open(MODELS_DIR / "feature_cols_v5.json") as f: _fcols = json.load(f)

def apply(model, df: pd.DataFrame, param: dict) -> pd.DataFrame:
    _load()
    CLASS_NAMES = ["Normal", "Baja", "Media", "Alta"]
    missing = [c for c in _fcols if c not in df.columns]
    if missing:
        raise ValueError(f"Features faltantes: {missing[:5]}...")
    X_native = df[_fcols].values.astype(np.float32)
    X_scaled = _scaler.transform(df[_fcols].values).astype(np.float32)
    out = _infer.predict(X_scaled, X_native)
    df = df.copy()
    df["pred_label"]  = out["labels"]
    df["pred_class"]  = [CLASS_NAMES[l] for l in out["labels"]]
    df["prob_normal"] = out["proba"][:, 0]
    df["prob_baja"]   = out["proba"][:, 1]
    df["prob_media"]  = out["proba"][:, 2]
    df["prob_alta"]   = out["proba"][:, 3]
    df["ensemble"]    = "D38_D40"
    return df
'''
    APPLY_PATH.write_text(APPLY_CODE.strip())
    print(f'  Regenerado: {APPLY_PATH}')

spec_a = importlib.util.spec_from_file_location('dsdl_apply_d38', APPLY_PATH)
dsdl_mod = importlib.util.module_from_spec(spec_a)
spec_a.loader.exec_module(dsdl_mod)

print(f'✅ dsdl_apply_d38.py cargado: {APPLY_PATH}')
print(f'   Tamaño: {APPLY_PATH.stat().st_size / 1024:.1f} KB')
print(f'   Función apply() : {dsdl_mod.apply}')

✅ dsdl_apply_d38.py cargado: /srv/work/models/dsdl_apply_d38.py
   Tamaño: 2.7 KB
   Función apply() : <function apply at 0x7fbacdec3380>


In [5]:
import pyarrow.parquet as pq

with open(MODELS_DIR / 'feature_cols_v5.json') as f:
    feature_cols = json.load(f)
n_features = len(feature_cols)

print(f'Features D40: {n_features} columnas')

TE_PATH = WORK_DIR / 'te_features_v6.parquet'
pf = pq.read_table(TE_PATH, columns=feature_cols)
df_sample = pf.slice(0, 500).to_pandas()

print(f'Muestra real: {len(df_sample)} ventanas  ×  {len(df_sample.columns)} features')
print(f'dtype sample: {df_sample.dtypes.unique()}')

t0 = time.perf_counter()
df_result = dsdl_mod.apply(None, df_sample.copy(), {})
elapsed_ms = (time.perf_counter() - t0) * 1000

print(f'\n=== Resultado apply() sobre {len(df_sample)} ventanas reales ===')
print(f'  Tiempo total    : {elapsed_ms:.1f} ms')
print(f'  ms / ventana    : {elapsed_ms / len(df_sample):.3f} ms')
print(f'  Columnas output : {[c for c in df_result.columns if c not in feature_cols]}')
print()

dist = df_result['pred_class'].value_counts()
print('Distribución de predicciones:')
for cls in CLASS_NAMES:
    n = dist.get(cls, 0)
    pct = n / len(df_result) * 100
    print(f'  {cls:<8} : {n:>5}  ({pct:4.1f}%)')
print()
print(f'Prob Alta max : {df_result["prob_alta"].max():.4f}')
print(f'Prob Alta p95 : {df_result["prob_alta"].quantile(0.95):.4f}')
print()
print('✅ apply() funcional sobre datos reales')

Features D40: 100 columnas
Muestra real: 500 ventanas  ×  100 features
dtype sample: [dtype('float64') dtype('int8') dtype('int32')]


/srv/python_packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/srv/python_packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/srv/python_packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. Fo


=== Resultado apply() sobre 500 ventanas reales ===
  Tiempo total    : 5978.4 ms
  ms / ventana    : 11.957 ms
  Columnas output : ['pred_label', 'pred_class', 'prob_normal', 'prob_baja', 'prob_media', 'prob_alta', 'ensemble']

Distribución de predicciones:
  Normal   :   448  (89.6%)
  Baja     :    50  (10.0%)
  Media    :     2  ( 0.4%)
  Alta     :     0  ( 0.0%)

Prob Alta max : 0.2162
Prob Alta p95 : 0.1030

✅ apply() funcional sobre datos reales


2026-10-13 14:18:19.167827898 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {500} for output label


In [6]:

LATENCY_SLA_MS = 100.0
N_WARMUP       = 3
N_REPS         = 8

pf_bench = pq.read_table(TE_PATH, columns=feature_cols)
df_bench = pf_bench.slice(0, 1000).to_pandas()

batch_sizes = [1, 10, 100, 500, 1_000]
latency_results = []

print(f'=== Benchmark de latencia D40 — datos reales (SLA < {LATENCY_SLA_MS} ms/ventana) ===')
print(f'  {"Batch":>8}  {"Total ms":>10}  {"ms/ventana":>12}  {"SLA":>6}  {"Throughput (v/s)":>18}')
print('  ' + '-' * 64)

for bs in batch_sizes:
    subset = df_bench.iloc[:bs].copy()
    for _ in range(N_WARMUP):
        _ = dsdl_mod.apply(None, subset, {})
    times = []
    for _ in range(N_REPS):
        t0 = time.perf_counter()
        _ = dsdl_mod.apply(None, subset, {})
        times.append((time.perf_counter() - t0) * 1000)
    total_ms  = np.median(times)
    per_win   = total_ms / bs
    sla_ok    = per_win < LATENCY_SLA_MS
    throughput = bs / (total_ms / 1000)
    latency_results.append({
        'batch_size': bs,
        'total_ms'  : round(total_ms, 2),
        'ms_per_win': round(per_win, 3),
        'sla_ok'    : bool(sla_ok),
        'throughput': round(throughput, 1),
    })
    sla_sym = '✅' if sla_ok else '❌'
    print(f'  {bs:>8,}  {total_ms:>10.2f}  {per_win:>12.3f}  {sla_sym:>6}  {throughput:>18,.0f}')

all_ok = all(r['sla_ok'] for r in latency_results)
print()
print(f'SLA < {LATENCY_SLA_MS} ms/ventana: {"✅ CUMPLIDO en todos los batches" if all_ok else "❌ VIOLADO — revisar"}')

=== Benchmark de latencia D40 — datos reales (SLA < 100.0 ms/ventana) ===
     Batch    Total ms    ms/ventana     SLA    Throughput (v/s)
  ----------------------------------------------------------------
         1       63.84        63.839       ✅                  16


2026-10-13 14:20:26.280780918 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-10-13 14:20:26.356684065 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-10-13 14:20:26.433017190 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-10-13 14:20:26.509769347 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-10-13 14:20:26.586662250 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-10-13 14:20:26.662439015 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} doe

        10       76.75         7.675       ✅                 130


2026-10-13 14:20:27.488672473 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {100} for output label
2026-10-13 14:20:27.583658028 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {100} for output label
2026-10-13 14:20:27.676623264 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {100} for output label
2026-10-13 14:20:27.782156584 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {100} for output label
2026-10-13 14:20:27.888597062 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {100} for output label
2026-10-13 14:20:27.995310725 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1

       100      104.65         1.047       ✅                 956


2026-10-13 14:20:28.469306134 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {500} for output label
2026-10-13 14:20:28.617861892 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {500} for output label
2026-10-13 14:20:28.748048002 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {500} for output label
2026-10-13 14:20:28.902005358 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {500} for output label
2026-10-13 14:20:29.054182688 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {500} for output label
2026-10-13 14:20:29.206805512 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1

       500      149.62         0.299       ✅               3,342


2026-10-13 14:20:30.143421726 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {1000} for output label
2026-10-13 14:20:30.390668351 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {1000} for output label
2026-10-13 14:20:30.599446183 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {1000} for output label
2026-10-13 14:20:30.798744073 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {1000} for output label
2026-10-13 14:20:30.982067000 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {1000} for output label
2026-10-13 14:20:31.194226744 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model 

     1,000      196.48         0.196       ✅               5,090

SLA < 100.0 ms/ventana: ✅ CUMPLIDO en todos los batches


2026-10-13 14:20:32.156870512 [W:onnxruntime:, execution_frame.cc:921 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {1000} for output label


In [7]:

SPL_QUERIES = {
    'Scoring en tiempo real (ventanas recientes)': f'''
index=dsdl sourcetype=latam2cloud_ml earliest=-15m
| fields rtt_ms_mean, loss_pct_mean, jitter_ms_mean, bw_mbps_mean,
         latency_p95, loss_p95, jitter_p95,
         as_path_len_mean, prefix_stability_30m, mttd_flag,
         hour_sin, hour_cos, dow_sin, dow_cos
| apply {MODEL_NAME}
| table _time, pred_class, prob_alta, prob_media, prob_baja, prob_normal, ensemble
| sort -_time
''',
    'Alerta threshold-based sobre predicciones DSDL': f'''
index=dsdl sourcetype=latam2cloud_ml earliest=-30m
| apply {MODEL_NAME}
| where prob_alta > 0.6
| stats count as n_alta, avg(prob_alta) as avg_prob_alta by region
| where n_alta > 5
| eval alerta="DEGRADACIÓN ALTA DETECTADA POR ML"
| table region, n_alta, avg_prob_alta, alerta
''',
    'Dashboard: comparar ML vs umbral estático (H3)': f'''
index=dsdl earliest=-24h
| eval source_type=if(sourcetype=="latam2cloud_alerts", "umbral_estatico", "ml_dsdl")
| search sourcetype IN (latam2cloud_alerts, latam2cloud_ml)
| eval is_alert_static=if(sourcetype=="latam2cloud_alerts" AND alerted=1, 1, 0)
| apply {MODEL_NAME}
| eval is_alert_ml=if(pred_class=="Alta" AND prob_alta > 0.5, 1, 0)
| stats sum(is_alert_static) as total_static, sum(is_alert_ml) as total_ml by date_hour
| eval mttd_reduction_pct=round((total_static - total_ml) / total_static * 100, 1)
| table date_hour, total_static, total_ml, mttd_reduction_pct
''',
    'Validación: apply() sobre lookup histórico': f'''
| inputlookup mttd_results_d38.csv
| fields rtt_ms_mean, loss_pct_mean, jitter_ms_mean, bw_mbps_mean,
         latency_p95, loss_p95, jitter_p95,
         as_path_len_mean, prefix_stability_30m, mttd_flag,
         hour_sin, hour_cos, dow_sin, dow_cos
| apply {MODEL_NAME}
| stats count by pred_class
'''
}

print(f'=== SPL de integración DSDL — modelo: {MODEL_NAME} ===')
print('(Requiere: modelo registrado en Splunk Model Management + datos en index=dsdl)')
print()
for title, spl in SPL_QUERIES.items():
    print(f'--- {title} ---')
    print(spl.strip())
    print()

=== SPL de integración DSDL — modelo: latam2cloud_monitor ===
(Requiere: modelo registrado en Splunk Model Management + datos en index=dsdl)

--- Scoring en tiempo real (ventanas recientes) ---
index=dsdl sourcetype=latam2cloud_ml earliest=-15m
| fields rtt_ms_mean, loss_pct_mean, jitter_ms_mean, bw_mbps_mean,
         latency_p95, loss_p95, jitter_p95,
         as_path_len_mean, prefix_stability_30m, mttd_flag,
         hour_sin, hour_cos, dow_sin, dow_cos
| apply latam2cloud_monitor
| table _time, pred_class, prob_alta, prob_media, prob_baja, prob_normal, ensemble
| sort -_time

--- Alerta threshold-based sobre predicciones DSDL ---
index=dsdl sourcetype=latam2cloud_ml earliest=-30m
| apply latam2cloud_monitor
| where prob_alta > 0.6
| stats count as n_alta, avg(prob_alta) as avg_prob_alta by region
| where n_alta > 5
| eval alerta="DEGRADACIÓN ALTA DETECTADA POR ML"
| table region, n_alta, avg_prob_alta, alerta

--- Dashboard: comparar ML vs umbral estático (H3) ---
index=dsdl earli

In [8]:

REGISTRATION_STEPS = f"""
=== Registro del modelo en Splunk AI Toolkit (DSDL) ===

Nombre del modelo : {MODEL_NAME}
Script DSDL       : {SCRIPT_PATH}
Entry point       : apply(model, df, param) — dsdl_apply_d38.py (nb09 Cell 15)
Ensemble          : RF v5.1×{W_RF} + LGB v7×{W_LGB} (LOG-25/LOG-30)

Opción A — Splunk Search Bar (| fit):
  | makeresults count=1
  | eval rtt_ms_mean=0.0, loss_pct_mean=0.0, jitter_ms_mean=0.0
  | fit {MODEL_NAME} rtt_ms_mean loss_pct_mean jitter_ms_mean into {MODEL_NAME}

Opción B — Splunk Web (AI Toolkit → Model Management):
  https://{SPLUNK_HOST}/en-US/app/Splunk_ML_Toolkit/modelmanagement

Verificación post-registro (Splunk Search Bar):
  index=dsdl sourcetype=latam2cloud_ml earliest=-15m
  | apply {MODEL_NAME}
  | stats count by pred_class
"""

print(REGISTRATION_STEPS)

import getpass
import splunklib.client as splunk_client
import splunklib.results as splunk_results

print(f'Conectando a Splunk REST API: https://{SPLUNK_HOST}:{SPLUNK_PORT}')
_splunk_user = input('Usuario Splunk [admin]: ').strip() or 'admin'
_splunk_pass = getpass.getpass('Contraseña Splunk: ')

try:
    s = splunk_client.connect(
        host=SPLUNK_HOST, port=SPLUNK_PORT,
        username=_splunk_user, password=_splunk_pass,
        scheme='https'
    )
    print(f'✅ Splunk REST API — Splunk {s.info["version"]}')

    try:
        idx_results = s.jobs.oneshot(
            'search index=dsdl sourcetype=latam2cloud_ml | head 1 | stats count',
            output_mode='json'
        )
        for row in splunk_results.JSONResultsReader(idx_results):
            if isinstance(row, dict):
                print(f'   ✅ Índice dsdl (latam2cloud_ml): {row.get("count", "0")} eventos')
    except Exception as e_idx:
        print(f'   ⚠️  Índice dsdl: {type(e_idx).__name__} — puede estar vacío o sin datos aún')

    try:
        model_check = s.jobs.oneshot(
            f'| rest /servicesNS/-/-/mltk/models | where title="{MODEL_NAME}"',
            output_mode='json'
        )
        found = False
        for row in splunk_results.JSONResultsReader(model_check):
            if isinstance(row, dict) and row.get('title') == MODEL_NAME:
                found = True
                print(f'   ✅ Modelo {MODEL_NAME} registrado en AI Toolkit')
                break
        if not found:
            print(f'   ⚠️  Modelo {MODEL_NAME} NO registrado aún en AI Toolkit')
            print(f'      → Ejecutar Opción A (| fit desde Splunk Search Bar)')
    except Exception as e_model:
        print(f'   ⚠️  Verificación de registro: {type(e_model).__name__}: {e_model}')

except Exception as e:
    print(f'ℹ️  Conexión REST fallida: {type(e).__name__}: {e}')
    print('   Verificar credenciales y acceso a Splunk.')
finally:
    del _splunk_pass



=== Registro del modelo en Splunk AI Toolkit (DSDL) ===

Nombre del modelo : latam2cloud_monitor
Script DSDL       : /srv/work/models/dsdl_apply_d38.py
Entry point       : apply(model, df, param) — dsdl_apply_d38.py (nb09 Cell 15)
Ensemble          : RF v5.1×0.8 + LGB v7×0.2 (LOG-25/LOG-30)

Opción A — Splunk Search Bar (| fit):
  | makeresults count=1
  | eval rtt_ms_mean=0.0, loss_pct_mean=0.0, jitter_ms_mean=0.0
  | fit latam2cloud_monitor rtt_ms_mean loss_pct_mean jitter_ms_mean into latam2cloud_monitor

Opción B — Splunk Web (AI Toolkit → Model Management):
  https://latam2cloud.iartificial.io/en-US/app/Splunk_ML_Toolkit/modelmanagement

Verificación post-registro (Splunk Search Bar):
  index=dsdl sourcetype=latam2cloud_ml earliest=-15m
  | apply latam2cloud_monitor
  | stats count by pred_class

Conectando a Splunk REST API: https://latam2cloud.iartificial.io:8089


Usuario Splunk [admin]:  admin
Contraseña Splunk:  ········


✅ Splunk REST API — Splunk 10.2.2
   ✅ Índice dsdl (latam2cloud_ml): 0 eventos
   ⚠️  Modelo latam2cloud_monitor NO registrado aún en AI Toolkit
      → Ejecutar Opción A (| fit desde Splunk Search Bar)


In [9]:
print('=' * 60)
print('RESUMEN OE3-A — Notebook 11 splunk_dsdl_apply v1')
print('=' * 60)
print()
print('Artefactos D40 utilizados (aprobados en etapas previas):')
print('  ensemble_d38_infer.py      ← nb08 (ONNX export)')
print('  lgb_v7.onnx                ← nb08 (ONNX export)')
print('  rf_v51_model.pkl           ← OE2 (training)')
print('  scaler_v5.pkl              ← OE2 (training)')
print('  feature_cols_v5.json       ← OE2 (training)')
print('  ensemble_d38_manifest.json ← nb09, LOG-30 (D40)')
print(f'  dsdl_apply_d38.py          ← nb09 Cell 15  →  {APPLY_PATH}  ✅')
print()
print('Verificaciones:')
print(f'  apply() sobre {len(df_sample)} ventanas reales         ✅')
print(f'  SLA < {LATENCY_SLA_MS} ms/ventana                    : {"✅ CUMPLIDO" if all_ok else "❌ VIOLADO"}')
print()
print('Latencia por batch (datos reales te_features_v6.parquet):')
for r in latency_results:
    sym = '✅' if r['sla_ok'] else '❌'
    print(f'  {r["batch_size"]:>6,} ventanas : {r["total_ms"]:>8.2f} ms total | '
          f'{r["ms_per_win"]:>8.3f} ms/ventana | {r["throughput"]:>10,.0f} v/s  {sym}')
print()
print('Splunk integration:')
print(f'  SPL      : | apply {MODEL_NAME}')
print(f'  Script   : {SCRIPT_PATH}')
print(f'  Modelos  : RF v5.1 × {W_RF} + LGB v7 × {W_LGB} (Ensemble D38 D40, LOG-25/LOG-30)')
print()
print('Estado OE3:')
print('  Camino A (DSDL live apply)  : ✅ Notebook 11  ← este notebook')
print('  Camino B (ML Toolkit custom): ➡️  Trabajo futuro (fuera de alcance tesis)')
print()
print('OE3 COMPLETADO: H3 VERIFICADA (MTTD_ML=8.7 min vs MTTD_estatico=30 min, −71%)')

RESUMEN OE3-A — Notebook 11 splunk_dsdl_apply v1

Artefactos D40 utilizados (aprobados en etapas previas):
  ensemble_d38_infer.py      ← nb08 (ONNX export)
  lgb_v7.onnx                ← nb08 (ONNX export)
  rf_v51_model.pkl           ← OE2 (training)
  scaler_v5.pkl              ← OE2 (training)
  feature_cols_v5.json       ← OE2 (training)
  ensemble_d38_manifest.json ← nb09, LOG-30 (D40)
  dsdl_apply_d38.py          ← nb09 Cell 15  →  /srv/work/models/dsdl_apply_d38.py  ✅

Verificaciones:
  apply() sobre 500 ventanas reales         ✅
  SLA < 100.0 ms/ventana                    : ✅ CUMPLIDO

Latencia por batch (datos reales te_features_v6.parquet):
       1 ventanas :    63.84 ms total |   63.839 ms/ventana |         16 v/s  ✅
      10 ventanas :    76.75 ms total |    7.675 ms/ventana |        130 v/s  ✅
     100 ventanas :   104.65 ms total |    1.047 ms/ventana |        956 v/s  ✅
     500 ventanas :   149.62 ms total |    0.299 ms/ventana |      3,342 v/s  ✅
   1,000 ventanas : 